<h3 style="color: #e0e0ff; font-style: italic;">🔍 SAFE: Similarity-Aware Multi-Modal Fake News Detection</h3>

---

**SAFE** (Similarity-Aware Multi-Modal Fake News Detection) is used here to capture cross-modal inconsistency:
- **Text Feature Extraction**: Claim text is encoded using BERT into a representation $H_t \in \mathbb{R}^{768}$.
- **Image Feature Extraction**: The image is encoded using ResNet-50 into a representation $H_v \in \mathbb{R}^{2048}$.
- **Projection**: Both representations are projected into a shared semantic space of dimension $d$ (e.g., 512):
  $$F_t = \text{ReLU}(W_t H_t + b_t), \quad F_v = \text{ReLU}(W_v H_v + b_v)$$
- **Similarity Computation**: The cosine similarity between $F_t$ and $F_v$ is calculated for each sample:
  $$\text{sim}(F_t, F_v) = \frac{F_t \cdot F_v}{\|F_t\|_2 \cdot \|F_v\|_2}$$
- **Feature Fusion**: The projected text features, projected image features, and the computed similarity score are concatenated:
  $$F_{\text{combined}} = [F_t, F_v, \text{sim}(F_t, F_v)] \in \mathbb{R}^{2d + 1}$$
- **Classification**: A fully connected classifier predicts the probability of the news being Fake or Real.

**Imports and Setup**

In [1]:
# Core
import time
import os
import random   
import json
from PIL import Image
from tqdm import tqdm

# Data manipulation
import numpy as np
import pandas as pd

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models

# Transformers
from transformers import XLMRobertaModel, XLMRobertaTokenizer

# sklearn
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, confusion_matrix, 
    roc_curve, auc, roc_auc_score, precision_recall_fscore_support
)

# Housekeeping
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

def set_seed(seed=42):
    random.seed(seed)  
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

os.makedirs('../models/safe', exist_ok=True)
os.makedirs('../reports/safe/figures', exist_ok=True)
os.makedirs('../reports/safe/metrics', exist_ok=True)
sns.set_palette("husl")

import sys
sys.path.insert(0, os.path.abspath("../src"))
from data_pipeline import M4FCDataPipeline

c:\Users\SAFAE ERAJI\Desktop\M2\Stage PFE\multimodal_fake_news_detection\venv_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Device: cuda


**Data Loading and Preparation**

In [2]:
class M4FCDataset(Dataset):
    def __init__(self, df, tokenizer, max_text_length=512, image_size=(224, 224)):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.image_size = image_size
        self.text_field = 'multilingual_claim'
            
        print(f"Using text field: {self.text_field}")
    
    def __len__(self):
        return len(self.df)
    
    def load_image(self, image_path):
        """Loading and preprocessing the image"""
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert('RGB')
                image = image.resize(self.image_size)
                # From NumPy image into PyTorch tensor in float format, and normalization of pixel values to the [0,1] range
                image = torch.from_numpy(np.array(image)).float().permute(2, 0, 1) / 255.0
                # Normalizing with ImageNet stats, because pre-trained models are trained on images normalized with specific mean and std values (ImageNet stats).
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                image = (image - mean) / std
                return image
            else:
                return torch.zeros(3, self.image_size[0], self.image_size[1])
        except Exception as e:
            return torch.zeros(3, self.image_size[0], self.image_size[1])
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Processing text
        text = str(row[self.text_field]) if pd.notna(row[self.text_field]) else ""
        encoding = self.tokenizer(
            text,
            max_length=self.max_text_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt' # for direct input into a transformer model
        )
        
        # Processing image
        image_path = row['full_image_path'] if pd.notna(row['full_image_path']) else ""
        if image_path.startswith("../"):
            image_path = "../" + image_path
        image = self.load_image(image_path)
        
        # Processing metadata 
        is_ai = float(row['is_ai_generated']) if pd.notna(row['is_ai_generated']) else 0.0
        img_exists = float(row['image_exists']) if pd.notna(row['image_exists']) else 0.0
        word_count = float(row['claim_word_count']) if pd.notna(row['claim_word_count']) else 0.0
        word_count = word_count / 50.0  # normalize
        manipulated = float(row['is_manipulated_fake']) if pd.notna(row['is_manipulated_fake']) else 0.0
        use_caption = float(row['use_true_caption']) if pd.notna(row['use_true_caption']) else 0.0
        
        metadata = torch.tensor([is_ai, img_exists, word_count, manipulated, use_caption], dtype=torch.float)
        
        # Get label
        label = torch.tensor(row['target'], dtype=torch.long)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'image': image,
            'metadata': metadata,
            'label': label,
            'text': text,
            'image_path': image_path
        }

In [3]:
print("📂 Loading pre-extracted features and CSV metadata...")

# Load all features
img_feats = torch.load('../data/features/image_features.pt')
text_feats = torch.load('../data/features/text_features.pt')
labels = torch.load('../data/features/labels.pt')
df = pd.read_csv('../data/M4FC.csv')

# Align all to minimum length
min_len = min(len(img_feats), len(text_feats), len(labels), len(df))
img_feats = img_feats[:min_len]
text_feats = text_feats[:min_len]
labels = labels[:min_len].long()


📂 Loading pre-extracted features and CSV metadata...


**SAFE Model Architecture**

In [4]:
class SAFEModel(nn.Module):
    """
    SAFE: Similarity-Aware Multi-Modal Fake News Detection.
    Enhanced with metadata support and weight initialization.
    
    Key features:
    - Frozen BERT (768-dim) + ResNet-50 (2048-dim) backbones
    - Projection to shared 512-dim space
    - Cosine similarity between text and image
    - Optional metadata integration
    - Xavier initialization
    """
    def __init__(self, text_encoder, image_encoder, proj_dim=512, num_classes=2, 
                 use_metadata=True, metadata_dim=5, dropout=0.3):
        super().__init__()
        self.text_encoder = text_encoder
        self.image_encoder = image_encoder
        self.use_metadata = use_metadata
        
        # Freeze backbones
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.image_encoder.parameters():
            param.requires_grad = False
            
        # Shared projection heads
        self.text_proj = nn.Sequential(
            nn.Linear(768, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.image_proj = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Metadata encoder (if used)
        if use_metadata:
            self.metadata_proj = nn.Sequential(
                nn.Linear(metadata_dim, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(dropout * 0.5)
            )
            classifier_input = proj_dim * 2 + 1 + 64  # text + image + similarity + metadata
        else:
            classifier_input = proj_dim * 2 + 1  # text + image + similarity
        
        # Similarity-Aware Classifier
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
        self._init_weights()
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"✅ SAFE model initialized")
        print(f"   Total params: {total_params:,} | Trainable: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
        print(f"   Metadata: {'Yes' if use_metadata else 'No'}")
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        
    def forward(self, input_ids, attention_mask, images, metadata=None):
        # Extract frozen features
        with torch.no_grad():
            text_feats = self.text_encoder(input_ids, attention_mask=attention_mask).pooler_output
            image_feats = self.image_encoder(images)
            
        # Project to shared space
        f_t = self.text_proj(text_feats)
        f_v = self.image_proj(image_feats)
        
        # Cosine similarity
        similarity = F.cosine_similarity(f_t, f_v, dim=-1).unsqueeze(1)
        
        # Concatenate
        if self.use_metadata and metadata is not None:
            meta_feats = self.metadata_proj(metadata)
            combined = torch.cat([f_t, f_v, similarity, meta_feats], dim=-1)
        else:
            combined = torch.cat([f_t, f_v, similarity], dim=-1)
        
        logits = self.classifier(combined)
        return logits, similarity

**Training Pipeline**

In [5]:
class SAFETrainer:
    def __init__(self, model, train_loader, val_loader, test_loader, 
                 device='cuda', save_name='SAFE', early_stopping_patience=10):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.save_name = save_name
        self.early_stopping_patience = early_stopping_patience
        
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=20)
        # ---------------------------------------------------------
        # PONDÉRATION DES CLASSES (Calcul robuste via le DataLoader)
        # ---------------------------------------------------------
        num_real = 0
        num_fake = 0
        for batch in self.train_loader:
            # Compatible avec les Dataset Dictionnaires (VisualBERT) ou Tuples (EANN/SpotFake)
            labels = batch['label'] if isinstance(batch, dict) else batch[2]
            num_real += (labels == 0).sum().item()
            num_fake += (labels == 1).sum().item()
            
        weight_real = 1.0 / num_real if num_real > 0 else 1.0
        weight_fake = 1.0 / num_fake if num_fake > 0 else 1.0
        weights = torch.tensor([weight_real, weight_fake], dtype=torch.float).to(device)
        weights = weights / weights.sum()
        
        self.criterion = nn.CrossEntropyLoss(weight=weights)
        # ---------------------------------------------------------
        
        self.metrics_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 
                                'val_acc': [], 'precision': [], 'recall': [], 
                                'f1': [], 'auc': []}
        self.best_val_f1 = 0.0
        self.best_val_f1 = 0.0
        self.best_val_acc = 0.0
        self.best_model_path = f"../models/safe/{save_name}_classifier_best.pth"
        os.makedirs(os.path.dirname(self.best_model_path), exist_ok=True)

    def train_epoch(self):
        self.model.train()
        total_loss, preds_all, labels_all = 0.0, [], []
        for batch in tqdm(self.train_loader, desc=f"Training {self.save_name}"):
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            image = batch['image'].to(self.device)
            metadata = batch['metadata'].to(self.device)
            lbl = batch['label'].to(self.device)
            
            self.optimizer.zero_grad()
            logits, _ = self.model(input_ids, attention_mask, image, metadata)
            loss = self.criterion(logits, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            total_loss += loss.item()
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            labels_all.extend(lbl.cpu().numpy())
        self.scheduler.step()
        return total_loss / len(self.train_loader), accuracy_score(labels_all, preds_all)

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total_loss, preds_all, probs_all, labels_all = 0.0, [], [], []
        for batch in loader:
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            image = batch['image'].to(self.device)
            metadata = batch['metadata'].to(self.device)
            lbl = batch['label'].to(self.device)
            
            logits, _ = self.model(input_ids, attention_mask, image, metadata)
            total_loss += self.criterion(logits, lbl).item()
            probs = F.softmax(logits, dim=1)
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            probs_all.extend(probs[:, 1].cpu().numpy())
            labels_all.extend(lbl.cpu().numpy())
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_all, preds_all, average='binary', zero_division=0)
        try:
            auc = roc_auc_score(labels_all, probs_all)
        except:
            auc = 0.5
            
        return {'loss': total_loss / len(loader), 'accuracy': accuracy_score(labels_all, preds_all),
                'precision': precision, 'recall': recall, 'f1': f1, 'auc': auc,
                'predictions': preds_all, 'true_labels': labels_all, 'probabilities': probs_all}

    def print_hardware_utilization(self):
        import psutil
        import torch
        cpu_percent = psutil.cpu_percent()
        ram = psutil.virtual_memory()
        print(f"\n💻 Hardware Utilization:")
        print(f"  - CPU: {cpu_percent}% | RAM: {ram.used / (1024**3):.2f} GB ({ram.percent}%)")
        if torch.cuda.is_available():
            print(f"  - GPU VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB allocated\n")

    def train(self, epochs=100):
        print(f"🚀 Training {self.save_name} | Patience={self.early_stopping_patience}")
        patience_counter = 0
        for epoch in range(epochs):
            train_loss, train_acc = self.train_epoch()
            val = self.evaluate(self.val_loader)
            
            self.metrics_history['train_loss'].append(train_loss)
            self.metrics_history['train_acc'].append(train_acc)
            self.metrics_history['val_loss'].append(val['loss'])
            self.metrics_history['val_acc'].append(val['accuracy'])
            self.metrics_history['precision'].append(val['precision'])
            self.metrics_history['recall'].append(val['recall'])
            self.metrics_history['f1'].append(val['f1'])
            self.metrics_history['auc'].append(val['auc'])
            
            print(f"Epoch {epoch+1:2d}/{epochs} | TL: {train_loss:.4f} | VL: {val['loss']:.4f} | "
                  f"VA: {val['accuracy']:.4f} | VF1: {val['f1']:.4f}")
            
            self.print_hardware_utilization()
            if val['f1'] > self.best_val_f1:
                self.best_val_f1 = val['f1']
                self.best_val_acc = val['accuracy']
                torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                           'val_accuracy': val['accuracy']}, self.best_model_path)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    print(f"🛑 Early stopping at epoch {epoch+1}")
                    break
        return self.metrics_history

    def test(self):
        if os.path.exists(self.best_model_path):
            self.model.load_state_dict(torch.load(self.best_model_path)['model_state_dict'])
        test_metrics = self.evaluate(self.test_loader)
        print(f"Test | Acc: {test_metrics['accuracy']:.4f} | F1: {test_metrics['f1']:.4f} | AUC: {test_metrics['auc']:.4f}")
        return test_metrics


**Evaluation and Visualizations**

In [6]:
class SAFEVisualizer:
    @staticmethod
    def plot_all(metrics_history, test_metrics, save_name):
        fig_dir = f"../reports/safe/figures/{save_name}"
        os.makedirs(fig_dir, exist_ok=True)
        
        # 1. Training History
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes[0, 0].plot(metrics_history['train_loss'], label='Train Loss')
        axes[0, 0].plot(metrics_history['val_loss'], label='Val Loss')
        axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
        axes[0, 1].plot(metrics_history['train_acc'], label='Train Acc')
        axes[0, 1].plot(metrics_history['val_acc'], label='Val Acc')
        axes[0, 1].set_title('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
        axes[1, 0].plot(metrics_history['f1'], label='F1', marker='o')
        axes[1, 0].plot(metrics_history['auc'], label='AUC', marker='s')
        axes[1, 0].set_title('F1 & AUC'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
        axes[1, 1].plot(metrics_history['precision'], label='Precision', marker='^')
        axes[1, 1].plot(metrics_history['recall'], label='Recall', marker='v')
        axes[1, 1].set_title('Precision-Recall'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(f"{fig_dir}/loss_accuracy_history.png", dpi=100); plt.close()
        
        # 2. Radar Chart
        cats = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
        vals = [test_metrics[k] for k in ['accuracy', 'precision', 'recall', 'f1', 'auc']]
        angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist() + [0]
        vals += [vals[0]]
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
        ax.plot(angles, vals, 'o-', lw=2, color='#e67e22'); ax.fill(angles, vals, alpha=0.25, color='#e67e22')
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats); ax.set_title(f'{save_name} Radar', size=14)
        plt.tight_layout(); plt.savefig(f"{fig_dir}/metrics_radar_chart.png", dpi=100); plt.close()
        
        # 3. Confusion Matrix
        cm = confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
        plt.title(f'Confusion Matrix - {save_name}'); plt.ylabel('True'); plt.xlabel('Predicted')
        plt.savefig(f"{fig_dir}/confusion_matrix.png", dpi=100); plt.close()
        
        # 4. ROC Curve
        fpr, tpr, _ = roc_curve(test_metrics['true_labels'], test_metrics['probabilities'])
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
        plt.plot([0, 1], [0, 1], '--', color='navy', lw=2); plt.xlim([0, 1]); plt.ylim([0, 1.05])
        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f'ROC - {save_name}')
        plt.legend(loc="lower right"); plt.grid(True, alpha=0.3)
        plt.savefig(f"{fig_dir}/roc_curve.png", dpi=100); plt.close()
        
        # 5. Prediction Distribution
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(test_metrics['probabilities'], bins=30, edgecolor='black', alpha=0.7, color='darkorange')
        axes[0].axvline(x=0.5, color='red', linestyle='--', lw=2, label='Threshold')
        axes[0].set_title('Prediction Distribution'); axes[0].set_xlabel('P(Fake)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].boxplot(test_metrics['probabilities'], vert=False, widths=0.5)
        axes[1].scatter(test_metrics['probabilities'], [1]*len(test_metrics['probabilities']), alpha=0.1, color='darkorange')
        axes[1].axvline(x=0.5, color='red', linestyle='--', lw=2)
        axes[1].set_title('Box Plot'); axes[1].set_xlabel('P(Fake)')
        plt.tight_layout(); plt.savefig(f"{fig_dir}/prediction_distribution.png", dpi=100); plt.close()
        
        print(f"✅ Plots saved to {fig_dir}")

**Model Saving & Reporting**

In [7]:
class SAFEModelSaver:
    @staticmethod
    def save_full_model(model, config, save_name):
        save_dir = f"../models/safe/{save_name.lower()}_model_saved"
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{save_dir}/model.pth")
        with open(f"{save_dir}/config.json", 'w') as f:
            json.dump(config, f, indent=2)
        size_mb = sum(p.numel() for p in model.parameters()) * 4 / (1024**2)
        print(f"✅ Model {save_name} saved ({size_mb:.2f} MB)")
        return size_mb
        
    @staticmethod
    def generate_report(metrics_history, test_metrics, save_name):
        path = f"../reports/safe/metrics/{save_name}_Report.txt"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        cm = confusion_matrix(test_metrics['true_labels'], test_metrics['predictions'])
        
        with open(path, 'w') as f:
            f.write(f"{'='*60}\n{save_name} FAKE NEWS DETECTION REPORT\n{'='*60}\n\n")
            f.write("TEST METRICS\n" + "-"*35 + "\n")
            f.write(f"Accuracy:  {test_metrics['accuracy']:.4f}\n")
            f.write(f"Precision: {test_metrics['precision']:.4f}\n")
            f.write(f"Recall:    {test_metrics['recall']:.4f}\n")
            f.write(f"F1 Score:  {test_metrics['f1']:.4f}\n")
            f.write(f"AUC:       {test_metrics['auc']:.4f}\n\n")
            f.write("CONFUSION MATRIX\n" + "-"*35 + "\n")
            f.write(f"TN={cm[0,0]}  FP={cm[0,1]}\nFN={cm[1,0]}  TP={cm[1,1]}\n\n")
            f.write("CONVERGENCE\n" + "-"*35 + "\n")
            f.write(f"Best Val Acc:     {max(metrics_history['val_acc']):.4f}\n")
            f.write(f"Final Train Acc:  {metrics_history['train_acc'][-1]:.4f}\n")
            f.write(f"Final Val Acc:    {metrics_history['val_acc'][-1]:.4f}\n")
            f.write(f"Final Train Loss: {metrics_history['train_loss'][-1]:.4f}\n")
            f.write(f"Final Val Loss:   {metrics_history['val_loss'][-1]:.4f}\n")
        print(f"✅ Report saved to {path}")

In [8]:
# 1. Initialize Tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# 2. Split the DataFrame using the pipeline defined in your project
pipeline = M4FCDataPipeline(csv_path='../data/M4FC.csv')
train_df, val_df, test_df = pipeline.split_dataframe(df)

print(f"📊 Splits -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 3. Create Dataset objects
train_dataset = M4FCDataset(train_df, tokenizer, max_text_length=512)
val_dataset = M4FCDataset(val_df, tokenizer, max_text_length=512)
test_dataset = M4FCDataset(test_df, tokenizer, max_text_length=512)

# 4. Create DataLoaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ DataLoaders successfully created!")

📊 Splits -> Train: 3462 | Val: 743 | Test: 743
Using text field: multilingual_claim
Using text field: multilingual_claim
Using text field: multilingual_claim
✅ DataLoaders successfully created!


**Run Training & Evaluation**

In [9]:
set_seed(42)

# Initialize Encoders
print("=" * 50)
print("📥 Loading BERT...")
text_encoder = XLMRobertaModel.from_pretrained('xlm-roberta-base')
for param in text_encoder.parameters():
    param.requires_grad = False
print("✅ BERT loaded & frozen")

print("📥 Loading ResNet-50...")
image_encoder = models.resnet50(pretrained=True)
image_encoder.fc = nn.Identity()
for param in image_encoder.parameters():
    param.requires_grad = False
print("✅ ResNet-50 loaded & frozen")

# Initialize Model
model = SAFEModel(text_encoder, image_encoder, proj_dim=512, use_metadata=True).to(device)
print(f"SAFE params: {sum(p.numel() for p in model.parameters()):,} total | "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable")

# Initialize Trainer
trainer = SAFETrainer(model, train_loader, val_loader, test_loader, 
                      device=device, save_name='SAFE', early_stopping_patience=10)

# Train
history = trainer.train(epochs=100)
test_metrics = trainer.test()

# Visualizations
SAFEVisualizer.plot_all(history, test_metrics, 'safe')

# Save
config = {
    "model": "SAFE",
    "text_encoder": "BERT-base (frozen)",
    "image_encoder": "ResNet-50 (frozen)",
    "text_dim": 768, "image_dim": 2048, "proj_dim": 512,
    "fusion": "concatenation + cosine similarity", "num_classes": 2,
    "metadata_used": True, "metadata_dim": 5,
    "dropout": 0.3, "classifier_dropout": 0.4,
    "optimizer": "AdamW", "lr": 1e-4, "weight_decay": 1e-4,
    "scheduler": "CosineAnnealingLR", "T_max": 20,
    "gradient_clipping": 1.0,
    "batch_size": 16, "max_epochs": 100, "actual_epochs": len(history['train_loss']),
    "early_stopping_patience": 10,
    "dataset": "M4FC", "split": "70/15/15", "seed": 42,
    "reference": "Zhou et al., 2020 (SAFE)"
}
SAFEModelSaver.save_full_model(model, config, 'SAFE')
SAFEModelSaver.generate_report(history, test_metrics, 'SAFE')

📥 Loading BERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1574.67it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT loaded & frozen
📥 Loading ResNet-50...
✅ ResNet-50 loaded & frozen
✅ SAFE model initialized
   Total params: 303,293,314 | Trainable: 1,741,634 (0.6%)
   Metadata: Yes
SAFE params: 303,293,314 total | 1,741,634 trainable
🚀 Training SAFE | Patience=10


Training SAFE: 100%|██████████| 217/217 [01:37<00:00,  2.22it/s]


Epoch  1/100 | TL: 0.8033 | VL: 23.3279 | VA: 0.9408 | VF1: 0.9695

💻 Hardware Utilization:
  - CPU: 21.3% | RAM: 22.52 GB (71.1%)
  - GPU VRAM: 1.17 GB allocated



...

Epoch  9/100 | TL: 0.9586 | VL: 23.0985 | VA: 0.9408 | VF1: 0.9695

💻 Hardware Utilization:
  - CPU: 19.6% | RAM: 21.98 GB (69.4%)
  - GPU VRAM: 1.17 GB allocated



Training SAFE: 100%|██████████| 217/217 [01:36<00:00,  2.25it/s]


Epoch 10/100 | TL: 1.0169 | VL: 22.0923 | VA: 0.9408 | VF1: 0.9695

💻 Hardware Utilization:
  - CPU: 18.8% | RAM: 22.09 GB (69.7%)
  - GPU VRAM: 1.17 GB allocated



Training SAFE: 100%|██████████| 217/217 [01:36<00:00,  2.24it/s]


Epoch 11/100 | TL: 0.9350 | VL: 23.2096 | VA: 0.9408 | VF1: 0.9695

💻 Hardware Utilization:
  - CPU: 22.4% | RAM: 22.15 GB (70.0%)
  - GPU VRAM: 1.17 GB allocated

🛑 Early stopping at epoch 11
Test | Acc: 0.9408 | F1: 0.9695 | AUC: 0.5000
✅ Plots saved to ../reports/safe/figures/safe
✅ Model SAFE saved (1156.97 MB)
✅ Report saved to ../reports/safe/metrics/SAFE_Report.txt
